In [2]:
from pydantic_ai import Agent
from toyaikit.tools import get_instance_methods
import agent_search_tools


tools = get_instance_methods(agent_search_tools)

instructions = """
You're a documentation assistant.

Answer the user question using only the documentation knowledge base.

Make 3 iterations:

1) First iteration:
   - Perform one search using the search tool to identify potentially relevant documents.
   - Explain (in 2â€“3 sentences) why this search query is appropriate for the user question.

2) Second iteration:
   - Analyze the results from the previous search.
   - Based on the filenames or documents returned, perform:
       - Up to 2 additional search queries to refine or expand coverage, and
       - One or more get_file calls to retrieve the full content of the most relevant documents.
   - For each search or get_file call, explain (in 2â€“3 sentences) why this action is necessary and how it helps answer the question.

3) Third iteration:
   - Analyze the retrieved document contents from get_file.
   - Synthesize the information from these documents into a final answer to the user.

IMPORTANT:
- At every step, explicitly explain your reasoning for each search query or file retrieval.
- Use only facts found in the documentation knowledge base.
- Do not introduce outside knowledge or assumptions.
- If the answer cannot be found in the retrieved documents, clearly inform the user.

Additional notes:
- The knowledge base is entirely about Evidently, so you do not need to include the word "evidently" in search queries.
- Prefer retrieving and analyzing full documents (via get_file) before producing the final answer.
"""

search_agent = Agent(
    name='search',
    model = 'openai:gpt-4o-mini',
    instructions=instructions,
    tools=tools
)

In [3]:
query = "how do I use evidently to monitor my machine learning models?"

result = await search_agent.run(query)

In [4]:
print(result.output)

1) First iteration:
   - I will search using the query "monitor machine learning models."
   - This search query is appropriate because it directly targets documentation that likely covers how to use the tool for monitoring models, which is the primary focus of your question.

2) Second iteration:
   - Analyzing the search results, I will proceed with two additional searches: "model monitoring" and "ML model evaluation." This will help refine my search to find more specific information related to monitoring machine learning models and evaluating their performance.
   - Additionally, I will perform a get_file call on a document that appears to be focused on monitoring, as it will provide detailed insights relevant to your question.

3) Third iteration:
   - After retrieving and analyzing the relevant documents about monitoring machine learning models, I will synthesize the information to provide a comprehensive answer.
   
Now I'll begin the first step to identify relevant documents. 



In [5]:
messages = result.all_messages()
result.usage()

RunUsage(input_tokens=321, output_tokens=199, details={'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, requests=1)

In [6]:
for m in messages:
    print(m.kind)
    for p in m.parts:
        part_kind = p.part_kind
        if part_kind == "user-prompt":
            print("USER:", p.content)
        if part_kind == "tool-call":
            print("TOOL CALL:", p.tool_name, p.args)
        if part_kind == "tool-return":
            print("TOOL RETURN:", p.tool_name)
        if part_kind == "text":
            print(p.content)
    print()

request
USER: how do I use evidently to monitor my machine learning models?

response
1) First iteration:
   - I will search using the query "monitor machine learning models."
   - This search query is appropriate because it directly targets documentation that likely covers how to use the tool for monitoring models, which is the primary focus of your question.

2) Second iteration:
   - Analyzing the search results, I will proceed with two additional searches: "model monitoring" and "ML model evaluation." This will help refine my search to find more specific information related to monitoring machine learning models and evaluating their performance.
   - Additionally, I will perform a get_file call on a document that appears to be focused on monitoring, as it will provide detailed insights relevant to your question.

3) Third iteration:
   - After retrieving and analyzing the relevant documents about monitoring machine learning models, I will synthesize the information to provide a comp

In [7]:
new_prompt = "show me the code"

result2 = await search_agent.run(new_prompt, message_history=messages)

In [8]:
print(result2.output)

1) First iteration:
   - I will search using the query "code for monitoring machine learning models."
   - This search query is appropriate because it should lead me to specific examples or code snippets that demonstrate how to implement monitoring for machine learning models, which is what you are looking for.

2) Second iteration:
   - Analyzing the results from the previous search, I will perform two additional searches: "model monitoring code example" and "ML monitoring script." This will allow me to gather more targeted examples and scripts that might detail the implementation of monitoring for models.
   - I will also initiate one or more get_file calls to retrieve full documents that include code examples, which will help provide the practical code implementation you need.

3) Third iteration:
   - After reviewing the content from the retrieved documents, I will synthesize the details to provide you with concrete code examples for monitoring machine learning models.

Now I'll be

In [9]:
print(result2.new_messages())

[ModelRequest(parts=[UserPromptPart(content='show me the code', timestamp=datetime.datetime(2026, 5, 2, 20, 18, 5, 843547, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 5, 2, 20, 18, 5, 844580, tzinfo=datetime.timezone.utc), instructions='You\'re a documentation assistant.\n\nAnswer the user question using only the documentation knowledge base.\n\nMake 3 iterations:\n\n1) First iteration:\n   - Perform one search using the search tool to identify potentially relevant documents.\n   - Explain (in 2â€“3 sentences) why this search query is appropriate for the user question.\n\n2) Second iteration:\n   - Analyze the results from the previous search.\n   - Based on the filenames or documents returned, perform:\n       - Up to 2 additional search queries to refine or expand coverage, and\n       - One or more get_file calls to retrieve the full content of the most relevant documents.\n   - For each search or get_file call, explain (in 2â€“3 sentences) why this action is 

In [ ]:
def print_messages(messages):
    for m in messages:
        print(m.kind)
        for p in m.parts:
            part_kind = p.part_kind
            if part_kind == "user-prompt":
                print("USER:", p.content)
            if part_kind == "tool-call":
                print("TOOL CALL:", p.tool_name, p.args)
            if part_kind == "tool-return":
                print("TOOL RETURN:", p.tool_name)
            if part_kind == "text":
                print(p.content)
        print()



In [11]:
print_messages(result2.new_messages())

request
USER: show me the code

response
1) First iteration:
   - I will search using the query "code for monitoring machine learning models."
   - This search query is appropriate because it should lead me to specific examples or code snippets that demonstrate how to implement monitoring for machine learning models, which is what you are looking for.

2) Second iteration:
   - Analyzing the results from the previous search, I will perform two additional searches: "model monitoring code example" and "ML monitoring script." This will allow me to gather more targeted examples and scripts that might detail the implementation of monitoring for models.
   - I will also initiate one or more get_file calls to retrieve full documents that include code examples, which will help provide the practical code implementation you need.

3) Third iteration:
   - After reviewing the content from the retrieved documents, I will synthesize the details to provide you with concrete code examples for monitor

In [12]:
from pydantic_ai import RunUsage


messages = []
usage = RunUsage()

while True:
    user_prompt = input("You:")
    if user_prompt.lower().strip() == "stop":
        break

    result = await search_agent.run(user_prompt, message_history=messages)
    usage = usage + result.usage()

    print_messages(result.new_messages())
    messages.extend(result.new_messages())

In [13]:
usage = RunUsage()
usage

RunUsage()

In [14]:
from pydantic_ai.messages import FunctionToolCallEvent


class NamedCallback:

    def __init__(self, agent):
        self.agent_name = agent.name

    async def print_function_calls(self, ctx, event):
        # Detect nested streams
        if hasattr(event, "__aiter__"):
            async for sub in event:
                await self.print_function_calls(ctx, sub)
            return

        if isinstance(event, FunctionToolCallEvent):
            tool_name = event.part.tool_name
            args = event.part.args
            print(f"TOOL CALL ({self.agent_name}): {tool_name}({args})")

    async def __call__(self, ctx, event):
        return await self.print_function_calls(ctx, event)

In [15]:
callback = NamedCallback(search_agent)

In [16]:
result = await search_agent.run(
    "how do I build dashboads with evidently?", event_stream_handler=callback
)